In [ ]:
# IND MODEL — Trend Prediction (STRICT MODEL-CONSISTENT)
# Full-data analysis
#   - full data
#   - scaled global trend
#   - NO trend wrapping
import numpy as np
import pyreadr
from tqdm import tqdm
from pathlib import Path
# Paths and settings
BASE_DIR = Path(r"path/to/snow/data-and-results")
period = 52
# 1. Load data (FULL)
snow = pyreadr.read_r(BASE_DIR / "snow_cleaned_full.Rda")
snow = list(snow.values())[0].reset_index(drop=True)

coords = snow.iloc[:, :2].to_numpy()
y = snow.iloc[:, 2:].to_numpy()

S, TT = y.shape
print("Using FULL S =", S, "TT =", TT)
# 2. Global trend scaling (HISTORICAL ONLY)
t_hist = np.arange(1, TT + 1)
mean_hist = t_hist.mean()
sd_hist = t_hist.std(ddof=0)
# 3. Load posterior (10 chains, thin=15)
thin = 15
n_chains = 10

theta01_list = []
theta10_list = []

for c in range(n_chains):
    d01 = np.load(BASE_DIR / f"p01_ind_all_chain{c}.npz")
    d10 = np.load(BASE_DIR / f"p10_ind_all_chain{c}.npz")

    key01 = d01.files[0]
    key10 = d10.files[0]

    theta01_list.append(d01[key01][:, ::thin])
    theta10_list.append(d10[key10][:, ::thin])

theta01 = np.concatenate(theta01_list, axis=1)
theta10 = np.concatenate(theta10_list, axis=1)

assert theta01.shape[0] == 4 * S
assert theta10.shape[0] == 4 * S

M = theta01.shape[1]
print("Loaded posterior samples M =", M)
# 4. Slice blocks
beta0_01 = theta01[0*S:1*S, :]
beta1_01 = theta01[1*S:2*S, :]
beta2_01 = theta01[2*S:3*S, :]
alpha_01 = theta01[3*S:4*S, :]

beta0_10 = theta10[0*S:1*S, :]
beta1_10 = theta10[1*S:2*S, :]
beta2_10 = theta10[2*S:3*S, :]
alpha_10 = theta10[3*S:4*S, :]
# 5. Allocate
weekly_ini = np.zeros((M, S, 52, 2))
weekly_final = np.zeros((M, S, 52, 2))

inv_logit = lambda x: 1 / (1 + np.exp(-x))

print("Begin IND trend prediction (strict, FULL data)...")
# MAIN LOOP
for m in tqdm(range(M), desc="Posterior samples"):

    b0 = beta0_01[:, m]
    b1 = beta1_01[:, m]
    b2 = beta2_01[:, m]
    a01 = alpha_01[:, m]

    b0s = beta0_10[:, m]
    b1s = beta1_10[:, m]
    b2s = beta2_10[:, m]
    a10 = alpha_10[:, m]

    init_state = np.column_stack([
        y[:, 0] == 0,
        y[:, 0] == 1
    ]).astype(float)
    # FIRST YEAR
    for week_idx in range(1, 53):

        curr = init_state.copy()

        if week_idx > 1:
            for t in range(1, week_idx):

                t_scaled = (t - mean_hist) / sd_hist

                eta01 = (
                    b0
                    + b1 * np.cos(2 * np.pi * t / period)
                    + b2 * np.sin(2 * np.pi * t / period)
                    + a01 * t_scaled
                )

                eta10 = (
                    b0s
                    + b1s * np.cos(2 * np.pi * t / period)
                    + b2s * np.sin(2 * np.pi * t / period)
                    + a10 * t_scaled
                )

                p01 = inv_logit(eta01)
                p10 = inv_logit(eta10)

                c0 = curr[:, 0]
                c1 = curr[:, 1]

                curr = np.column_stack([
                    c0 * (1 - p01) + c1 * p10,
                    c0 * p01 + c1 * (1 - p10)
                ])

        weekly_ini[m, :, week_idx - 1, :] = curr
    # FINAL YEAR (52 years ahead)
    curr = init_state.copy()
    total_steps = 52 * 52

    for t in range(1, total_steps + 1):

        t_scaled = (t - mean_hist) / sd_hist

        eta01 = (
            b0
            + b1 * np.cos(2 * np.pi * t / period)
            + b2 * np.sin(2 * np.pi * t / period)
            + a01 * t_scaled
        )

        eta10 = (
            b0s
            + b1s * np.cos(2 * np.pi * t / period)
            + b2s * np.sin(2 * np.pi * t / period)
            + a10 * t_scaled
        )

        p01 = inv_logit(eta01)
        p10 = inv_logit(eta10)

        c0 = curr[:, 0]
        c1 = curr[:, 1]

        curr = np.column_stack([
            c0 * (1 - p01) + c1 * p10,
            c0 * p01 + c1 * (1 - p10)
        ])

        if t > 51 * 52:
            w = t - 51 * 52 - 1
            weekly_final[m, :, w, :] = curr
# SAVE
np.savez_compressed(
    BASE_DIR / "trend_ind_full.npz",
    weekly_ini=weekly_ini,
    weekly_final=weekly_final
)

print("IND FULL-data prediction finished.")

In [ ]:
# BYM WEEKLY MODEL — Trend Prediction (STRICT MODEL-CONSISTENT)
# Full-data analysis
#   - full data
#   - scaled global trend
#   - weekly tau
#   - NO trend wrapping
import numpy as np
import pyreadr
from tqdm import tqdm
from pathlib import Path
import pickle
# Paths and settings
BASE_DIR = Path(r"path/to/snow/data-and-results")
period = 52
# 1. Load data (FULL)
snow = pyreadr.read_r(BASE_DIR / "snow_cleaned_full.Rda")
snow = list(snow.values())[0].reset_index(drop=True)

coords = snow.iloc[:, :2].to_numpy()
y = snow.iloc[:, 2:].to_numpy()

S, TT = y.shape
print("Using FULL S =", S, "TT =", TT)
# 2. Historical trend scaling
t_hist = np.arange(1, TT + 1)
mean_hist = t_hist.mean()
sd_hist = t_hist.std(ddof=0)
# 3. Load posterior (10 chains, thin=15)
thin = 15
n_chains = 10

eta01_list = []
tau01_list = []
eta10_list = []
tau10_list = []

for c in range(n_chains):

    with open(BASE_DIR / f"p01_weekly_chain{c}.pkl", "rb") as f:
        d = pickle.load(f)
    eta01_list.append(d["eta"][:, ::thin])
    tau01_list.append(d["tau"][:, ::thin])

    with open(BASE_DIR / f"p10_weekly_chain{c}.pkl", "rb") as f:
        d = pickle.load(f)
    eta10_list.append(d["eta"][:, ::thin])
    tau10_list.append(d["tau"][:, ::thin])

all_eta01 = np.concatenate(eta01_list, axis=1)
all_tau01 = np.concatenate(tau01_list, axis=1)

all_eta10 = np.concatenate(eta10_list, axis=1)
all_tau10 = np.concatenate(tau10_list, axis=1)

K_total = 8

assert all_eta01.shape[0] == K_total * S
assert all_eta10.shape[0] == K_total * S
assert all_tau01.shape[0] == K_total * 52
assert all_tau10.shape[0] == K_total * 52

M = all_eta01.shape[1]
print("Loaded posterior samples M =", M)
# 4. Allocate
weekly_ini = np.zeros((M, S, 52, 2))
weekly_final = np.zeros((M, S, 52, 2))

inv_logit = lambda x: 1 / (1 + np.exp(-x))

print("Begin BYM weekly trend prediction (strict, FULL data)...")
# MAIN LOOP
for m in tqdm(range(M), desc="Posterior samples"):

    eta01 = all_eta01[:, m]
    tau01 = all_tau01[:, m]

    eta10 = all_eta10[:, m]
    tau10 = all_tau10[:, m]

    eta01_mat = eta01.reshape(K_total, S)
    tau01_mat = tau01.reshape(K_total, 52)

    eta10_mat = eta10.reshape(K_total, S)
    tau10_mat = tau10.reshape(K_total, 52)

    init_state = np.column_stack([
        y[:, 0] == 0,
        y[:, 0] == 1
    ]).astype(float)
    # FIRST YEAR
    for week_idx in range(1, 53):

        curr = init_state.copy()

        if week_idx > 1:
            for t in range(1, week_idx):

                t_scaled = (t - mean_hist) / sd_hist
                w = (t - 1) % 52

                x_vec = np.array([
                    1.0,
                    np.cos(2 * np.pi * t / period),
                    np.sin(2 * np.pi * t / period),
                    t_scaled
                ])

                x_full = np.repeat(x_vec, 2)

                psi01 = np.zeros(S)
                psi10 = np.zeros(S)

                for k in range(K_total):
                    psi01 += x_full[k] * eta01_mat[k] * tau01_mat[k, w]
                    psi10 += x_full[k] * eta10_mat[k] * tau10_mat[k, w]

                p01 = inv_logit(psi01)
                p10 = inv_logit(psi10)

                c0 = curr[:, 0]
                c1 = curr[:, 1]

                curr = np.column_stack([
                    c0 * (1 - p01) + c1 * p10,
                    c0 * p01 + c1 * (1 - p10)
                ])

        weekly_ini[m, :, week_idx - 1, :] = curr
    # FINAL YEAR (52 years ahead)
    curr = init_state.copy()
    total_steps = 52 * 52

    for t in range(1, total_steps + 1):

        t_scaled = (t - mean_hist) / sd_hist
        w = (t - 1) % 52

        x_vec = np.array([
            1.0,
            np.cos(2 * np.pi * t / period),
            np.sin(2 * np.pi * t / period),
            t_scaled
        ])

        x_full = np.repeat(x_vec, 2)

        psi01 = np.zeros(S)
        psi10 = np.zeros(S)

        for k in range(K_total):
            psi01 += x_full[k] * eta01_mat[k] * tau01_mat[k, w]
            psi10 += x_full[k] * eta10_mat[k] * tau10_mat[k, w]

        p01 = inv_logit(psi01)
        p10 = inv_logit(psi10)

        c0 = curr[:, 0]
        c1 = curr[:, 1]

        curr = np.column_stack([
            c0 * (1 - p01) + c1 * p10,
            c0 * p01 + c1 * (1 - p10)
        ])

        if t > 51 * 52:
            w_store = t - 51 * 52 - 1
            weekly_final[m, :, w_store, :] = curr
# SAVE
np.savez_compressed(
    BASE_DIR / "trend_bym_weekly_full.npz",
    weekly_ini=weekly_ini,
    weekly_final=weekly_final
)

print("BYM weekly FULL-data prediction finished.")

In [ ]:
# BYM WEEKLY + FACTOR
# STRICT MODEL-CONSISTENT TREND PREDICTION
import numpy as np
import pyreadr
import geopandas as gpd
from scipy.spatial.distance import pdist, squareform
from scipy.sparse import csr_matrix
from scipy.sparse.csgraph import connected_components
from tqdm import tqdm
from pathlib import Path
import pickle
import pandas as pd
# Paths and settings
BASE_DIR = Path(r"path/to/snow/data-and-results")
DIST_TH = 0.22
period = 52

snow = pyreadr.read_r(BASE_DIR/"snow_cleaned_full.Rda")
snow = list(snow.values())[0].reset_index(drop=True)

coords = snow.iloc[:, :2].to_numpy()
y = snow.iloc[:, 2:].to_numpy()

S, TT = y.shape
print("Using FULL S =", S)
# 3. Historical trend scaling
t_hist = np.arange(1, TT+1)
mean_hist = t_hist.mean()
sd_hist   = t_hist.std(ddof=0)
# 4. Load covariates (scaled exactly as fitting)
# latitude
lat_raw = coords[:,1]
lat = (lat_raw - lat_raw.mean()) / lat_raw.std()

# elevation
no_nbs = np.array([
    57,170,236,269,343,685,946,947,989,
    1037,1084,1090,1109,1118,1127,1176,1203
]) - 1

elev_raw = pd.read_csv(BASE_DIR/"curr_elev.csv").iloc[:,3].to_numpy()
nnbs_elev = pd.read_csv(BASE_DIR/"nnbs_elev.csv", sep="\t").iloc[:,2].to_numpy()

elev_all = np.zeros(S)
mask = np.ones(S, dtype=bool)
mask[no_nbs] = False

elev_all[mask] = elev_raw
elev_all[no_nbs] = nnbs_elev

elev = (elev_all - elev_all.mean()) / elev_all.std()


# temperature (global scaling as in MCMC)
snow_temp = pyreadr.read_r(BASE_DIR/"snow_temp_full.Rda")
snow_temp = list(snow_temp.values())[0].reset_index(drop=True)
temp_full = snow_temp.iloc[:,2:].to_numpy()
temp_scaled = (temp_full - temp_full.mean()) / temp_full.std()
# 5. Load posterior (10 chains, thin=15)
thin = 15
n_chains = 10

eta01_list = []
tau01_list = []
eta10_list = []
tau10_list = []

for c in range(n_chains):

    with open(BASE_DIR/f"p01_weekly_cov_chain{c}.pkl","rb") as f:
        d = pickle.load(f)

    eta01_list.append(d["eta"][:, ::thin])
    tau01_list.append(d["tau"][:, ::thin])

    with open(BASE_DIR/f"p10_weekly_cov_chain{c}.pkl","rb") as f:
        d = pickle.load(f)

    eta10_list.append(d["eta"][:, ::thin])
    tau10_list.append(d["tau"][:, ::thin])

all_eta01 = np.concatenate(eta01_list, axis=1)
all_tau01 = np.concatenate(tau01_list, axis=1)

all_eta10 = np.concatenate(eta10_list, axis=1)
all_tau10 = np.concatenate(tau10_list, axis=1)

M = all_eta01.shape[1]
print("Loaded posterior samples M =", M)

K_base = 4
K_total = 8
# 6. Allocate
weekly_ini   = np.zeros((M, S, 52, 2))
weekly_final = np.zeros((M, S, 52, 2))

inv_logit = lambda x: 1 / (1 + np.exp(-x))

print("Begin BYM weekly + factor trend prediction (strict)...")
# MAIN LOOP
for m in tqdm(range(M), desc="Posterior samples"):

    eta01 = all_eta01[:, m]
    tau01 = all_tau01[:, m]

    eta10 = all_eta10[:, m]
    tau10 = all_tau10[:, m]

    # spatial blocks
    eta01_sp = eta01[:K_total*S].reshape(K_total, S)
    eta10_sp = eta10[:K_total*S].reshape(K_total, S)

    tau01_mat = tau01.reshape(K_total, 52)
    tau10_mat = tau10.reshape(K_total, 52)

    # gamma (interaction-only)
    gamma01 = eta01[K_total*S:]
    gamma10 = eta10[K_total*S:]

    init_state = np.column_stack([
        y[:,0] == 0,
        y[:,0] == 1
    ]).astype(float)
    # FIRST YEAR
    for week_idx in range(1, 53):

        curr = init_state.copy()

        if week_idx > 1:
            for t in range(1, week_idx):

                t_scaled = (t - mean_hist) / sd_hist
                w = (t-1) % 52

                x_vec = np.array([
                    1.0,
                    np.cos(2*np.pi*t/period),
                    np.sin(2*np.pi*t/period),
                    t_scaled
                ])

                x_full = np.repeat(x_vec, 2)

                psi01 = np.zeros(S)
                psi10 = np.zeros(S)

                for k in range(K_total):
                    psi01 += x_full[k] * eta01_sp[k] * tau01_mat[k, w]
                    psi10 += x_full[k] * eta10_sp[k] * tau10_mat[k, w]

                # interaction-only factor
                temp_t = temp_scaled[:, (t-1) % TT]

                psi01 += t_scaled * (
                    gamma01[0] * lat
                    + gamma01[1] * elev
                    + gamma01[2] * temp_t
                )

                psi10 += t_scaled * (
                    gamma10[0] * lat
                    + gamma10[1] * elev
                    + gamma10[2] * temp_t
                )

                p01 = inv_logit(psi01)
                p10 = inv_logit(psi10)

                c0 = curr[:,0]
                c1 = curr[:,1]

                curr = np.column_stack([
                    c0*(1-p01) + c1*p10,
                    c0*p01     + c1*(1-p10)
                ])

        weekly_ini[m,:,week_idx-1,:] = curr
    # FINAL YEAR
    curr = init_state.copy()
    total_steps = 52*52

    for t in range(1, total_steps+1):

        t_scaled = (t - mean_hist) / sd_hist
        w = (t-1) % 52

        x_vec = np.array([
            1.0,
            np.cos(2*np.pi*t/period),
            np.sin(2*np.pi*t/period),
            t_scaled
        ])

        x_full = np.repeat(x_vec, 2)

        psi01 = np.zeros(S)
        psi10 = np.zeros(S)

        for k in range(K_total):
            psi01 += x_full[k] * eta01_sp[k] * tau01_mat[k, w]
            psi10 += x_full[k] * eta10_sp[k] * tau10_mat[k, w]

        temp_t = temp_scaled[:, (t-1) % TT]

        psi01 += t_scaled * (
            gamma01[0] * lat
            + gamma01[1] * elev
            + gamma01[2] * temp_t
        )

        psi10 += t_scaled * (
            gamma10[0] * lat
            + gamma10[1] * elev
            + gamma10[2] * temp_t
        )

        p01 = inv_logit(psi01)
        p10 = inv_logit(psi10)

        c0 = curr[:,0]
        c1 = curr[:,1]

        curr = np.column_stack([
            c0*(1-p01) + c1*p10,
            c0*p01     + c1*(1-p10)
        ])

        if t > 51*52:
            w_store = t - 51*52 - 1
            weekly_final[m,:,w_store,:] = curr
# SAVE
np.savez_compressed(
    BASE_DIR/"trend_weekly_bym+cov.npz",
    weekly_ini=weekly_ini,
    weekly_final=weekly_final
)

print("BYM weekly + factor prediction finished (strict).")

In [ ]:
# BYM WEEKLY + FACTOR + LONGITUDE
# STRICT MODEL-CONSISTENT TREND PREDICTION
import numpy as np
import pyreadr
from tqdm import tqdm
from pathlib import Path
import pickle
import pandas as pd
# Paths and settings
BASE_DIR = Path(r"path/to/snow/data-and-results")
period = 52

snow = pyreadr.read_r(BASE_DIR/"snow_cleaned_full.Rda")
snow = list(snow.values())[0].reset_index(drop=True)

coords = snow.iloc[:, :2].to_numpy()
y = snow.iloc[:, 2:].to_numpy()

S, TT = y.shape
print("Using FULL S =", S)
# 1. time scaling (historical)
t_hist = np.arange(1, TT+1)
mean_hist = t_hist.mean()
sd_hist   = t_hist.std(ddof=0)
# 2. covariates (MUST match MCMC)
# -------- longitude split --------
lon_raw = coords[:, 0]

region = np.zeros(S, dtype=int)
region[lon_raw < -30] = 0
region[lon_raw >= -30] = 1

lon_na = np.zeros(S)
lon_euas = np.zeros(S)

mask_na = (region == 0)
mask_euas = (region == 1)

lon_na[mask_na] = (
    (lon_raw[mask_na] - lon_raw[mask_na].mean()) /
    max(lon_raw[mask_na].std(), 1e-6)
)

lon_euas[mask_euas] = (
    (lon_raw[mask_euas] - lon_raw[mask_euas].mean()) /
    max(lon_raw[mask_euas].std(), 1e-6)
)

# -------- latitude --------
lat_raw = coords[:,1]
lat = (lat_raw - lat_raw.mean()) / lat_raw.std()

# -------- elevation --------
no_nbs = np.array([
    57,170,236,269,343,685,946,947,989,
    1037,1084,1090,1109,1118,1127,1176,1203
]) - 1

elev_raw = pd.read_csv(BASE_DIR/"curr_elev.csv").iloc[:,3].to_numpy()
nnbs_elev = pd.read_csv(BASE_DIR/"nnbs_elev.csv", sep="\t").iloc[:,2].to_numpy()

elev_all = np.zeros(S)
mask = np.ones(S, dtype=bool)
mask[no_nbs] = False

elev_all[mask] = elev_raw
elev_all[no_nbs] = nnbs_elev

elev = (elev_all - elev_all.mean()) / elev_all.std()

# -------- temperature --------
snow_temp = pyreadr.read_r(BASE_DIR/"snow_temp_full.Rda")
snow_temp = list(snow_temp.values())[0].reset_index(drop=True)
temp_full = snow_temp.iloc[:,2:].to_numpy()
temp_scaled = (temp_full - temp_full.mean()) / temp_full.std()
# 3. load posterior (10 chains, thin=15)
thin = 15
n_chains = 10

eta01_list, tau01_list = [], []
eta10_list, tau10_list = [], []

for c in range(n_chains):

    with open(BASE_DIR/f"p01_weekly_cov+lon_chain{c}.pkl","rb") as f:
        d = pickle.load(f)
    eta01_list.append(d["eta"][:, ::thin])
    tau01_list.append(d["tau"][:, ::thin])

    with open(BASE_DIR/f"p10_weekly_cov+lon_chain{c}.pkl","rb") as f:
        d = pickle.load(f)
    eta10_list.append(d["eta"][:, ::thin])
    tau10_list.append(d["tau"][:, ::thin])

all_eta01 = np.concatenate(eta01_list, axis=1)
all_tau01 = np.concatenate(tau01_list, axis=1)
all_eta10 = np.concatenate(eta10_list, axis=1)
all_tau10 = np.concatenate(tau10_list, axis=1)

M = all_eta01.shape[1]
print("Loaded posterior samples M =", M)

K_total = 8
# 4. allocate
weekly_ini   = np.zeros((M, S, 52, 2))
weekly_final = np.zeros((M, S, 52, 2))

inv_logit = lambda x: 1 / (1 + np.exp(-x))
# MAIN LOOP
print("Begin trend prediction (BYM + longitude)...")

for m in tqdm(range(M), desc="Posterior samples"):

    eta01 = all_eta01[:, m]
    tau01 = all_tau01[:, m]

    eta10 = all_eta10[:, m]
    tau10 = all_tau10[:, m]

    eta01_sp = eta01[:K_total*S].reshape(K_total, S)
    eta10_sp = eta10[:K_total*S].reshape(K_total, S)

    tau01_mat = tau01.reshape(K_total, 52)
    tau10_mat = tau10.reshape(K_total, 52)

    # gamma = 5
    gamma01 = eta01[K_total*S:]
    gamma10 = eta10[K_total*S:]

    init_state = np.column_stack([
        y[:,0] == 0,
        y[:,0] == 1
    ]).astype(float)

    # ================= FIRST YEAR =================
    for week_idx in range(1, 53):

        curr = init_state.copy()

        if week_idx > 1:
            for t in range(1, week_idx):

                t_scaled = (t - mean_hist) / sd_hist
                w = (t-1) % 52

                x_vec = np.array([
                    1.0,
                    np.cos(2*np.pi*t/period),
                    np.sin(2*np.pi*t/period),
                    t_scaled
                ])
                x_full = np.repeat(x_vec, 2)

                psi01 = np.zeros(S)
                psi10 = np.zeros(S)

                for k in range(K_total):
                    psi01 += x_full[k] * eta01_sp[k] * tau01_mat[k, w]
                    psi10 += x_full[k] * eta10_sp[k] * tau10_mat[k, w]

                temp_t = temp_scaled[:, (t-1) % TT]

                # ===== LONGITUDE VERSION =====
                psi01 += t_scaled * (
                    gamma01[0] * lon_na
                    + gamma01[1] * lon_euas
                    + gamma01[2] * lat
                    + gamma01[3] * elev
                    + gamma01[4] * temp_t
                )

                psi10 += t_scaled * (
                    gamma10[0] * lon_na
                    + gamma10[1] * lon_euas
                    + gamma10[2] * lat
                    + gamma10[3] * elev
                    + gamma10[4] * temp_t
                )

                p01 = inv_logit(psi01)
                p10 = inv_logit(psi10)

                c0 = curr[:,0]
                c1 = curr[:,1]

                curr = np.column_stack([
                    c0*(1-p01) + c1*p10,
                    c0*p01     + c1*(1-p10)
                ])

        weekly_ini[m,:,week_idx-1,:] = curr

    # ================= FINAL YEAR =================
    curr = init_state.copy()
    total_steps = 52*52

    for t in range(1, total_steps+1):

        t_scaled = (t - mean_hist) / sd_hist
        w = (t-1) % 52

        x_vec = np.array([
            1.0,
            np.cos(2*np.pi*t/period),
            np.sin(2*np.pi*t/period),
            t_scaled
        ])
        x_full = np.repeat(x_vec, 2)

        psi01 = np.zeros(S)
        psi10 = np.zeros(S)

        for k in range(K_total):
            psi01 += x_full[k] * eta01_sp[k] * tau01_mat[k, w]
            psi10 += x_full[k] * eta10_sp[k] * tau10_mat[k, w]

        temp_t = temp_scaled[:, (t-1) % TT]

        psi01 += t_scaled * (
            gamma01[0] * lon_na
            + gamma01[1] * lon_euas
            + gamma01[2] * lat
            + gamma01[3] * elev
            + gamma01[4] * temp_t
        )

        psi10 += t_scaled * (
            gamma10[0] * lon_na
            + gamma10[1] * lon_euas
            + gamma10[2] * lat
            + gamma10[3] * elev
            + gamma10[4] * temp_t
        )

        p01 = inv_logit(psi01)
        p10 = inv_logit(psi10)

        c0 = curr[:,0]
        c1 = curr[:,1]

        curr = np.column_stack([
            c0*(1-p01) + c1*p10,
            c0*p01     + c1*(1-p10)
        ])

        if t > 51*52:
            w_store = t - 51*52 - 1
            weekly_final[m,:,w_store,:] = curr
# SAVE
np.savez_compressed(
    BASE_DIR/"trend_weekly_bym+cov+lon.npz",
    weekly_ini=weekly_ini,
    weekly_final=weekly_final
)

print("Trend prediction finished (BYM + longitude).")

In [ ]:
# FINAL TRACEPLOT VERSION (PAPER-READY, NO SAVE)
# Full-data analysis
import numpy as np
import pandas as pd
import pickle
import pyreadr
import matplotlib.pyplot as plt

from pathlib import Path
# Paths and settings
BASE_DIR = Path(r"path/to/snow/data-and-results")

period = 52
thin2 = 15

locations = [70, 1400]
weeks = [20, 35]
year = 20

times = [(year * 52 + w - 1) for w in weeks]
chains = list(range(10))

no_nbs = np.array([
    57,170,236,269,343,685,946,947,989,
    1037,1084,1090,1109,1118,1127,1176,1203
]) - 1
# Data (FULL)
def load_data():
    snow = pyreadr.read_r(BASE_DIR / "snow_cleaned_full.Rda")
    snow = list(snow.values())[0].reset_index(drop=True)

    coords = snow.iloc[:, :2].to_numpy()
    y = snow.iloc[:, 2:].to_numpy()

    return coords, y

coords, y = load_data()
S, TT = y.shape

print("Using FULL S =", S, "TT =", TT)
# COVARIATES (FULL, EXACTLY AS FITTING)
lat = (coords[:, 1] - coords[:, 1].mean()) / coords[:, 1].std()

elev_raw = pd.read_csv(BASE_DIR / "curr_elev.csv").iloc[:, 3].to_numpy()
nnbs_elev = pd.read_csv(BASE_DIR / "nnbs_elev.csv", sep="\t").iloc[:, 2].to_numpy()

elev_all = np.zeros(S)
mask = np.ones(S, dtype=bool)
mask[no_nbs] = False

elev_all[mask] = elev_raw
elev_all[no_nbs] = nnbs_elev
elev = (elev_all - elev_all.mean()) / elev_all.std()

snow_temp = pyreadr.read_r(BASE_DIR / "snow_temp_full.Rda")
snow_temp = list(snow_temp.values())[0].reset_index(drop=True)
temp_full = snow_temp.iloc[:, 2:].to_numpy()
temp_scaled = (temp_full - temp_full.mean()) / temp_full.std()

t_full = np.arange(1, TT + 1)
t_scaled_full = (t_full - t_full.mean()) / t_full.std(ddof=0)

assert len(elev_raw) == mask.sum()
assert len(nnbs_elev) == len(no_nbs)
assert temp_scaled.shape[0] == S
assert temp_scaled.shape[1] == TT
# LOAD CHAINS
def load_bym(prefix):
    eta_list, tau_list = [], []

    for c in chains:
        fname = f"{prefix}_chain{c}.pkl"

        with open(BASE_DIR / fname, "rb") as f:
            d = pickle.load(f)

        eta_list.append(d["eta"][:, ::thin2])
        tau_list.append(d["tau"][:, ::thin2])

    return eta_list, tau_list


def load_ind(prefix):
    eta_list = []

    for c in chains:
        d = np.load(BASE_DIR / f"{prefix}_chain{c}.npz")
        key = d.files[0]
        eta_list.append(d[key][:, ::thin2])

    return eta_list
# POSTERIOR
def compute_p_bym(results_eta, results_tau, use_cov=True):
    K_total = 8

    if use_cov:
        assert results_eta[0].shape[0] == K_total * S + 3
    else:
        assert results_eta[0].shape[0] == K_total * S

    assert results_tau[0].shape[0] == K_total * 52

    results = {}

    for s in locations:
        for t in times:
            w = t % 52
            t_scaled = t_scaled_full[t]

            cov4 = np.array([
                1.0,
                np.cos(2 * np.pi * (t + 1) / period),
                np.sin(2 * np.pi * (t + 1) / period),
                t_scaled
            ])

            chains_out = []

            for c in range(len(results_eta)):
                eta = results_eta[c]
                tau = results_tau[c]

                phi = np.zeros(eta.shape[1])

                for k in range(K_total):
                    phi += eta[k * S + s, :] * tau[k * 52 + w, :] * cov4[k // 2]

                if use_cov:
                    gamma = eta[K_total * S: K_total * S + 3, :]
                    phi += gamma[0] * t_scaled * lat[s]
                    phi += gamma[1] * t_scaled * elev[s]
                    phi += gamma[2] * t_scaled * temp_scaled[s, t]

                p = 1 / (1 + np.exp(-phi))
                p_final = p if y[s, t] == 0 else 1 - p

                chains_out.append(p_final)

            results[(s, t)] = chains_out

    return results


def compute_p_ind(eta_list):
    K = 4

    assert eta_list[0].shape[0] == K * S

    results = {}

    for s in locations:
        for t in times:
            t_scaled = t_scaled_full[t]

            cov4 = np.array([
                1.0,
                np.cos(2 * np.pi * (t + 1) / period),
                np.sin(2 * np.pi * (t + 1) / period),
                t_scaled
            ])

            chains_out = []

            for c in range(len(eta_list)):
                eta = eta_list[c]
                phi = np.zeros(eta.shape[1])

                for k in range(K):
                    phi += eta[k * S + s, :] * cov4[k]

                p = 1 / (1 + np.exp(-phi))
                p_final = p if y[s, t] == 0 else 1 - p

                chains_out.append(p_final)

            results[(s, t)] = chains_out

    return results
# PLOT
def plot_model_subplots(results, model_name):

    fig, axs = plt.subplots(len(locations), len(times), figsize=(10, 6))
    axs = np.atleast_2d(axs)

    colors = plt.cm.tab10.colors

    for i, s in enumerate(locations):
        for j, t in enumerate(times):

            chain_samples = results[(s, t)]
            ax = axs[i, j]

            for c, chain in enumerate(chain_samples):

                ax.plot(
                    chain,
                    lw=0.6,
                    alpha=0.7,
                    color=colors[c % 10],
                    label=f"chain {c}" if (i==0 and j==0) else None
                )

            ax.set_title(f"{model_name}\nloc{s}, week{t % 52 + 1}", fontsize=9)
            ax.set_ylim(0, 1)

            ax.spines["top"].set_visible(False)
            ax.spines["right"].set_visible(False)
            ax.grid(alpha=0.25)

            ax.set_xlabel("")
            ax.set_ylabel("")

    handles, labels = axs[0,0].get_legend_handles_labels()
    fig.legend(
        handles,
        labels,
        loc="upper center",
        ncol=len(chain_samples),
        frameon=False,
        fontsize=9
    )

    plt.tight_layout(rect=[0, 0, 1, 0.92])
    plt.show()


def plot_gamma_subplots(eta_list, model_name):
    K_total = 8
    assert eta_list[0].shape[0] == K_total * S + 3

    cov_names = ["Latitude", "Elevation", "Temperature"]
    fig, axs = plt.subplots(3, 1, figsize=(10, 7))
    colors = plt.cm.tab10.colors

    for k in range(3):
        gamma_chains = []
        for c in range(len(eta_list)):
            gamma_chains.append(eta_list[c][K_total * S + k, :])

        ax = axs[k]

        for c, chain in enumerate(gamma_chains):
            ax.plot(
                chain,
                lw=0.6,
                alpha=0.7,
                color=colors[c % 10],
                label=f"chain {c}" if k == 0 else None
            )

        ax.set_title(f"{model_name} - {cov_names[k]}", fontsize=10)
        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)
        ax.grid(alpha=0.25)

    handles, labels = axs[0].get_legend_handles_labels()
    fig.legend(
        handles,
        labels,
        loc="upper center",
        ncol=len(gamma_chains),
        frameon=False,
        fontsize=9
    )

    plt.tight_layout(rect=[0, 0, 1, 0.93])
    plt.show()
# RUN
print("Loading chains...")

eta10_cov, tau10_cov = load_bym("p10_weekly_cov")
eta10, tau10 = load_bym("p10_weekly")
eta_ind10 = load_ind("p10_ind_all")

assert eta10_cov[0].shape[0] == 8 * S + 3
assert tau10_cov[0].shape[0] == 8 * 52
assert eta10[0].shape[0] == 8 * S
assert tau10[0].shape[0] == 8 * 52
assert eta_ind10[0].shape[0] == 4 * S

print("Computing posterior...")

res_cov = compute_p_bym(eta10_cov, tau10_cov, use_cov=True)
res_bym = compute_p_bym(eta10, tau10, use_cov=False)
res_ind = compute_p_ind(eta_ind10)

print("Plotting...")

plot_model_subplots(res_cov, "Weekly BYM + Covariates")
plot_model_subplots(res_bym, "Weekly BYM")
plot_model_subplots(res_ind, "Independent Model")

eta01_cov, _ = load_bym("p01_weekly_cov")
assert eta01_cov[0].shape[0] == 8 * S + 3

plot_gamma_subplots(eta01_cov, "BYM + Cov (01)")
plot_gamma_subplots(eta10_cov, "BYM + Cov (10)")

print("Done.")

In [ ]:
from pathlib import Path
import numpy as np

BASE_DIR = Path(r"path/to/snow/data-and-results")

def compute_trend(npz_file, out_file):
    
    data = np.load(BASE_DIR / npz_file)
    
    ini   = data["weekly_ini"]
    final = data["weekly_final"]
    
    ini_sum   = np.sum(ini[:,:,:,1], axis=2)
    final_sum = np.sum(final[:,:,:,1], axis=2)
    
    dpd = (final_sum - ini_sum) / 51
    
    mean  = np.quantile(dpd, 0.5, axis=0)
    sd    = dpd.std(axis=0)
    lower = np.quantile(dpd, 0.025, axis=0)
    upper = np.quantile(dpd, 0.975, axis=0)
    
    np.savez(BASE_DIR / out_file,
             mean=mean,
             sd=sd,
             lower=lower,
             upper=upper)

# compute_trend("trend_ind_full.npz", "trend_ind_summary.npz")
# compute_trend("trend_bym_weekly_full.npz", "trend_bym_summary.npz")
# compute_trend("trend_weekly_bym+cov.npz", "trend_bymp_summary.npz")
compute_trend("trend_weekly_bym+cov+lon.npz", "trend_bymp+lon_summary.npz")

In [ ]:
import numpy as np
import pickle
from pathlib import Path

BASE_DIR = Path(r"path/to/snow/data-and-results")

thin = 15
n_chains = 10

gamma01_list = []
gamma10_list = []

for c in range(n_chains):

    with open(BASE_DIR/f"p01_weekly_cov+lon_chain{c}.pkl","rb") as f:
        d = pickle.load(f)
    eta01 = d["eta"][:, ::thin]

    with open(BASE_DIR/f"p10_weekly_cov+lon_chain{c}.pkl","rb") as f:
        d = pickle.load(f)
    eta10 = d["eta"][:, ::thin]

    # Infer S (set this value to 5 here)
    eta_dim = eta01.shape[0]
    S = (eta_dim - 5) // 8

    gamma01_list.append(eta01[8*S:, :])  # (5, M)
    gamma10_list.append(eta10[8*S:, :])

# concat chains
gamma01 = np.concatenate(gamma01_list, axis=1)
gamma10 = np.concatenate(gamma10_list, axis=1)
# summary function
def summarize(x):
    return {
        "mean":   np.mean(x),
        "median": np.median(x),
        "lower":  np.quantile(x, 0.025),
        "upper":  np.quantile(x, 0.975)
    }
# names
names = [
    r"$\gamma_1$ (lon\_NA)",
    r"$\gamma_2$ (lon\_EUAS)",
    r"$\gamma_3$ (lat)",
    r"$\gamma_4$ (elev)",
    r"$\gamma_5$ (temp)",

    r"$\tilde{\gamma}_1$ (lon\_NA)",
    r"$\tilde{\gamma}_2$ (lon\_EUAS)",
    r"$\tilde{\gamma}_3$ (lat)",
    r"$\tilde{\gamma}_4$ (elev)",
    r"$\tilde{\gamma}_5$ (temp)"
]
# collect results
rows = []

for i in range(5):
    s = summarize(gamma01[i])
    rows.append((names[i], s))

for i in range(5):
    s = summarize(gamma10[i])
    rows.append((names[i+5], s))
# print LaTeX
print(r"\begin{table}[htbp]")
print(r"\centering")
print(r"\begin{tabular}{lcccc}")
print(r"\hline")
print(r"Parameter & Mean & Median & 2.5\% & 97.5\% \\")
print(r"\hline")

for name, s in rows:
    print(f"{name} & {s['mean']:.4f} & {s['median']:.4f} & {s['lower']:.4f} & {s['upper']:.4f} \\\\")

print(r"\hline")
print(r"\end{tabular}")
print(r"\caption{Posterior summaries of covariate effects including longitude interactions.}")
print(r"\end{table}")

In [ ]:
import pickle
import numpy as np
import os

BASE_DIR = r"path/to/snow/data-and-results"

prefix_list = ["p01_weekly", "p10_weekly"]

for prefix in prefix_list:
    for c in range(10):
        print(f"Converting {prefix}_chain{c}")

        pkl_path = os.path.join(BASE_DIR, f"{prefix}_chain{c}.pkl")
        npz_path = os.path.join(BASE_DIR, f"{prefix}_chain{c}.npz")

        with open(pkl_path, "rb") as f:
            d = pickle.load(f)

        eta = d["eta"]
        tau = d["tau"]
        np.savez(
            npz_path,
            eta=eta,
            tau=tau
        )

print("DONE")

In [ ]:
import numpy as np
import pickle
import pyreadr
import pandas as pd
from pathlib import Path

BASE_DIR = Path(r"path/to/snow/data-and-results")

thin = 15

def convert(prefix):
    for c in range(10):
        print(f"Processing {prefix} chain {c}")
        
        with open(BASE_DIR / f"{prefix}_chain{c}.pkl", "rb") as f:
            d = pickle.load(f)
        
        eta = d["eta"][:, ::thin]
        tau = d["tau"][:, ::thin]
        
        eta_df = pd.DataFrame(eta)
        tau_df = pd.DataFrame(tau)

        pyreadr.write_rds(
            BASE_DIR / f"{prefix}_eta_chain{c}.rds",
            eta_df
        )
        
        pyreadr.write_rds(
            BASE_DIR / f"{prefix}_tau_chain{c}.rds",
            tau_df
        )

convert("p01_weekly_cov+lon")
convert("p10_weekly_cov+lon")